# AfriVoices — Mesure des audios >20s (TEST par langue)

Le modèle n'a JAMAIS vu de clips >20s à l'entraînement (filtre). Si le test en contient
beaucoup, il décroche dessus. On mesure la proportion par langue → décide si un
**découpage en fenêtres** à l'inférence vaut le coup.

Autonome. Lit les parquets du test (Drive ou local).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import glob, os, io, base64, numpy as np
import soundfile as sf
import pandas as pd

BASE="/content/drive/MyDrive/afrivoices"
# localiser le test
cands=[f"{BASE}/test", "/content/test_data"]
TEST_DIR=next((d for d in cands if glob.glob(f"{d}/**/*.parquet", recursive=True)), None)
if TEST_DIR is None:
    print("test absent -> téléchargement HF...")
    from huggingface_hub import snapshot_download
    TEST_DIR="/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
parquets=sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")

In [ ]:
def duree(b):
    try: i=sf.info(io.BytesIO(b)); return i.frames/i.samplerate
    except Exception:
        try: i=sf.info(io.BytesIO(base64.b64decode(b))); return i.frames/i.samplerate
        except Exception: return None

from collections import defaultdict
durs=defaultdict(list)   # lang -> [durées]
from tqdm.auto import tqdm
for pq in tqdm(parquets, desc="scan test", unit="pq"):
    df=pd.read_parquet(pq)
    for _, r in df.iterrows():
        lang=r.get("language")
        b=r["audio"].get("bytes") if isinstance(r["audio"],dict) else r["audio"]
        d=duree(b)
        if d is not None: durs[lang].append(d)

print("\n=== proportion d'audios >20s par langue (TEST) ===")
print(f"{'lang':6} {'n':>7} {'>20s':>7} {'%>20s':>7} {'>30s':>7} {'durée max':>10} {'durée moy':>10}")
tot_n=tot_long=0
for lang in sorted(durs):
    ds=np.array(durs[lang])
    n=len(ds); long20=(ds>20).sum(); long30=(ds>30).sum()
    tot_n+=n; tot_long+=long20
    print(f"{str(lang):6} {n:>7} {long20:>7} {long20/n:>6.1%} {long30:>7} {ds.max():>9.1f}s {ds.mean():>9.1f}s")
print(f"\nGLOBAL: {tot_long}/{tot_n} = {tot_long/tot_n:.1%} des clips test dépassent 20s")

## Interprétation

- Si **>15-20 % sur kln/mas/luo** (langues à beaucoup d'unscripted) → le découpage en
  fenêtres de ~15s à l'inférence peut aider (le modèle reste dans son régime <20s).
- Si **<5 % partout** → négligeable, le découpage n'en vaut pas la peine.

Découpage en fenêtres (si le levier vaut le coup) : audio >20s → fenêtres de 15s avec
1s de recouvrement → transcrire chaque fenêtre → recoller les textes. À implémenter dans
le générateur de soumission (uniquement pour les clips longs).